<a href="https://colab.research.google.com/github/Git-Hub-Ran/rag-qa-langchain/blob/Dev/rag_qa_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "langchain==0.3.27" "langchain-community>=0.3.0" "langchain-core>=0.3.0" "langchain-openai>=0.3.0" langchain-text-splitters chromadb unstructured python-dotenv

In [2]:
#Connect to Google Drive to use API key:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
#Loading API key from .env:
from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/Colab Notebooks/openai-chatbot/.env")
print("OpenAI key loaded:", os.getenv("OPENAI_API_KEY") is not None) #Making sure the key was loaded

OpenAI key loaded: True


In [13]:
from langchain_community.document_loaders import UnstructuredURLLoader

# Webites that will be saved as documents:
urls = [
    "https://www.governor.ny.gov/news?items_per_page=10&current_page=%2Fnode%2F164161&created_date_1=2026-03-05&created_date=2026-03-05", #New York state news website on 5.3.26
    "https://www.gov.uk/government/news/government-launches-good-food-cycle-to-transform-britains-food-system" #Federal Government Press Release
]

#load documents:
loader = UnstructuredURLLoader(urls=urls)
docs = loader.load()

print(f"Number of documents that were loaded: {len(docs)}")

Number of documents that were loaded: 2


In [5]:
docs[0].page_content[:5000]  # Validation that the document was loaded : see the content of the document

'Pressroom\n\nOfficial news from the Office of the Governor\n\nMarch 05, 2026\n\nNew York Ranks High for Kids’ Online Safety Laws\n\nThe Anxious Generation Movement spotlighted Governor Hochul’s nation-leading work to protect kids online by curbing addictive feeds, regulating AI, and limiting phones in schools.\n\nProtecting Kids Online\n\nFeatured News\n\nTransformative I-81 Viaduct Project Updates\n\nTransformative I-81 Viaduct Project Updates\n\nMarch 5, 2026 | 3:16 PM EST\n\nGovernor Hochul announced the opening of Interstate 81 southbound as part of the transformative I-81 Viaduct Project, the largest infrastructure project in the history of the State Department of Transportation.\n\nHighlighting the Impact of Universal Child Care\n\nHighlighting the Impact of Universal Child Care\n\nMarch 5, 2026 | 2:59 PM EST\n\nGovernor Hochul underscored the importance of her plan to deliver universal child care for children under five years old across the state by highlighting the link betwee

In [6]:
#Break the docs into smaller chunks:
from langchain_text_splitters import RecursiveCharacterTextSplitter #This text splitter is the recommended one for generic text.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,#The maximum size of a chunk
    chunk_overlap=100 #Target overlap between chunks. Overlapping chunks helps to mitigate loss of information when context is divided between chunks.
    )
chunks = text_splitter.split_documents(docs)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 23


In [7]:
#create embedding:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

#creating vector store with Chroma:
from langchain_community.vectorstores import Chroma

vector_store = Chroma.from_documents(
    documents=chunks, #every chunk in the list
    embedding=embeddings #the function that convert every chunk into a numeric vector
)
#Now the vector store is ready for semantic search

In [8]:
#Create a retriever from the vector store. The retriever can get a question and return an answer from the relevent chunks of documents:
retriever = vector_store.as_retriever( search_kwargs={"k": 3})

In [9]:
from langchain_openai import ChatOpenAI

#Creating LLM:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [14]:
# Import the original RAG chain tools:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

queries_governor = [
    "What issue did Governor Hochul highlight regarding Trump's tariffs?",
    "What infrastructure project involving Interstate 81 was announced?",
    "What is the capital of France?" #This info doesn't exists in this website , so he should answer he don't know
]

queries_food_press = [
    "What is the main goal of the 'Good Food Cycle' initiative?",
    "Which outcomes does the Good Food Cycle framework identify?",
    "Why does the government consider food security important according to the press release?",
    "What is the capital of France?" #This info doesn't exists in this website , so he should answer he don't know
]

#define the prompt:
prompt = ChatPromptTemplate.from_messages([
   ("system",
    "You are a helpful assistant.\n"
    "Answer the question ONLY using the provided context.\n"
    "If the answer is not in the context, say 'I don’t know.'\n\n"
    "Context:\n{context}"
   ),
   ("human", "{input}")
])

# Create the answer-generation chain (LLM + prompt that processes retrieved documents):
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# Build complete retrieval augmented generation (RAG) pipeline:
qa_chain = create_retrieval_chain(retriever, question_answer_chain)

In [15]:
# Get answers on the Governor website:
for query in queries_governor:
    result = qa_chain.invoke({"input": query})

    print(f"\nQ: {query}")
    print(f"A: {result['answer']}\n")

    print("Retrieved Chunks:")
    for i, doc in enumerate(result["context"]):
        print(f"\nChunk {i+1}:")
        print(doc.page_content[:500])  # only part of the texts
        print("-" * 40)


Q: What issue did Governor Hochul highlight regarding Trump's tariffs?
A: Governor Hochul highlighted the issue of the Trump administration owing $13.5 billion in tariff payments to New Yorkers, which amounts to an estimated $1,751 per household.

Retrieved Chunks:

Chunk 1:
Governor Kathy Hochul New York State Seal

Press Release

Governor Hochul Announces Two Agency Appointments

Mar 5, 2026 | 01:54 PM EST

The Governor nominated Terry O’Leary to serve as Acting Commissioner of DHSES, and Doris B. González to serve as Acting President of HESC.

Governor Kathy Hochul New York State Seal

Press Release

Video, Audio, Photos & Rush Transcript: Governor Hochul Calls on Trump Administration to Refund $13.5 Billion Tariff Payments Owed to New Yorkers and Put Their Money
----------------------------------------

Chunk 2:
Highlighting the Impact of Universal Child Care

March 5, 2026 | 2:59 PM EST

Governor Hochul underscored the importance of her plan to deliver universal child care for ch

In [16]:
# Get answers on the Food Press release website:
for query in queries_food_press:
    result = qa_chain.invoke({"input": query})

    print(f"\nQ: {query}")
    print(f"A: {result['answer']}\n")

    print("Retrieved Chunks:")
    for i, doc in enumerate(result["context"]):
        print(f"\nChunk {i+1}:")
        print(doc.page_content[:500])  # only part of the texts
        print("-" * 40)


Q: What is the main goal of the 'Good Food Cycle' initiative?
A: The main goal of the 'Good Food Cycle' initiative is to drive a generational change in the nation’s relationship with food by promoting healthier eating, stronger food security, and greener supply chains.

Retrieved Chunks:

Chunk 1:
Press release

Government launches "Good Food Cycle" to transform Britain's food system

New "Good Food Cycle" framework serves up healthier eating, stronger food security and greener supply chains

The government has served up its new “Good Food Cycle” today (15 July) – a recipe aimed at driving a generational change in the nation’s relationship with food.

The Good Food Cycle identifies ten priority outcomes needed to build a thriving food sector while tackling challenges from rising obesity r
----------------------------------------

Chunk 2:
Minister for Health Ashley Dalton, said:

We want to make sure all families have the option of healthy, high-quality food – not least because it hel

We can see that the model answers questions using only the provided documents. It gives correct answers when information is in the text and says "I don’t know" when it is not.

Chunks show where the information came from.